# Walkthrough: one jammed PUSCH slot

This notebook follows one uplink slot through the whole simulation, step by step, in the style of
the Sionna tutorials:

1. the scheduled UE builds its PUSCH slot,
2. the slot goes through a TDL fading channel,
3. a jammer adds Gaussian noise on chosen PRB x symbol cells, through its own channel,
4. the gNB adds thermal noise and runs its receiver (channel estimation, equalization, decoding),
5. the three jamming features (power, DMRS residual, EVM) are computed from the receiver's results.

Each step uses the same functions as the project code in `src/jamloc/`.

In [ ]:
import matplotlib.pyplot as plt
import torch
from sionna.phy import config

from jamloc.config import SimConfig
from jamloc.jammer import ATTACKS, Jammer, cell_mask
from jamloc.ue import UEPool, carrier_resource_grid, place_on_carrier, take_allocation

cfg = SimConfig()
config.seed = 42
print("Device:", config.device)
print(cfg)

## 1. The scheduled UE

The UE is scheduled on 12 PRBs starting at PRB 4. Sionna's PUSCH transmitter always fills its whole
bandwidth part, so we build the slot for 12 PRBs and then place it on the 24-PRB carrier.

In [ ]:
alloc_start, alloc_size = 4, 12
tx = UEPool(cfg).transmitter(alloc_size)

x, b = tx(1)  # one slot: x is the frequency-domain resource grid, b the information bits
print("UE grid:", tuple(x.shape), " [batch, tx, antennas, OFDM symbols, subcarriers]")
print("Transport block:", b.shape[-1], "bits")

x_carrier = place_on_carrier(x, alloc_start, cfg)
print("Carrier grid:", tuple(x_carrier.shape))

plt.figure(figsize=(8, 3))
plt.imshow(x_carrier[0, 0, 0].abs().T.cpu(), aspect="auto", origin="lower", interpolation="nearest")
plt.xlabel("OFDM symbol"); plt.ylabel("Subcarrier"); plt.colorbar(label="|x|")
plt.title("UE resource grid on the carrier (DMRS on symbol 2, boosted by 3 dB)");

## 2. The UE's fading channel

3GPP TDL-A with 100 ns delay spread, speed up to 3 m/s. `OFDMChannel` applies the channel directly
on the resource grid (perfect synchronization, cyclic prefix longer than the channel).

In [ ]:
from sionna.phy.channel import AWGN, OFDMChannel
from sionna.phy.channel.tr38901 import TDL

rg = carrier_resource_grid(cfg)
ue_channel = OFDMChannel(TDL("A", 100e-9, cfg.carrier_frequency, max_speed=cfg.max_speed), rg,
                         normalize_channel=True)
jammer_channel = OFDMChannel(TDL("A", 100e-9, cfg.carrier_frequency, max_speed=cfg.max_speed), rg,
                             normalize_channel=True)

y_ue = ue_channel(x_carrier)  # no noise yet

## 3. The jammer

A pilot-targeted attack on PRBs 10-17: Gaussian noise only on the DMRS symbol. The jammer does not
know the allocation, so PRBs 16 and 17 fall outside it. The jammed cells are the labels.

In [ ]:
attack = torch.tensor([ATTACKS.index("pilot")])
labels = cell_mask(attack, jam_start=torch.tensor([10]), jam_size=torch.tensor([8]), cfg=cfg)

jsr_db = torch.tensor([0.0])  # jammer energy per jammed RE = UE energy per data RE
z = Jammer(cfg)(labels, jsr_db)
y_jam = jammer_channel(z)

plt.figure(figsize=(4, 3))
plt.imshow(labels[0].cpu(), aspect="auto", origin="lower", cmap="Greys")
plt.xlabel("OFDM symbol"); plt.ylabel("PRB"); plt.title("Jammed cells (label)");

## 4. Thermal noise and the gNB receiver

The gNB knows the allocation and the thermal noise variance, but not the jammer. The receiver is
Sionna's PUSCH chain written out: LS channel estimation, LMMSE equalization, demapping, decoding.

In [ ]:
from jamloc.receiver import PUSCHReceiverChain

snr_db = 15.0
no = torch.tensor([10 ** (-snr_db / 10)], device=config.device)
y = AWGN()(y_ue + y_jam, no)

y_alloc = take_allocation(y, alloc_start, alloc_size)
rx = PUSCHReceiverChain(tx)
out = rx(y_alloc, no)

print("Equalized data symbols:", tuple(out["x_hat"].shape))
print("Transport block CRC ok:", bool(out["tb_crc"].item()))
print("Bit errors:", int((out["b_hat"] != b).sum()))

plt.figure(figsize=(4, 4))
xh = out["x_hat"][0].cpu()
plt.scatter(xh.real, xh.imag, s=2, alpha=0.4)
plt.gca().set_aspect("equal"); plt.title("Equalized symbols (QPSK)");

## 5. The same slot through `JammedPUSCHLink`, and the features

`JammedPUSCHLink` runs steps 1-4 in one call for a whole batch. `extract` turns the receiver's
results into the 6-channel input of the detector: power, DMRS residual and EVM in dB, plus the
masks that say where the residual and EVM are defined.

In [ ]:
from jamloc.features import FEATURE_NAMES, extract
from jamloc.link import JammedPUSCHLink, Scenario, seed_everything

seed_everything(42)
n = len(ATTACKS)
scenario = Scenario(channel="A", alloc_start=4, alloc_size=12, attack=torch.arange(n),
                    jam_start=torch.full((n,), 10), jam_size=torch.full((n,), 8),
                    jsr_db=torch.full((n,), 0.0), snr_db=torch.full((n,), 15.0))
batch = JammedPUSCHLink(cfg)(scenario)
features = extract(batch, cfg)
print("Features:", tuple(features.shape), FEATURE_NAMES)

fig, axes = plt.subplots(n, 4, figsize=(10, 9), sharex=True, sharey=True)
for i, name in enumerate(ATTACKS):
    maps = [batch.labels[i].float(), features[i, 0], features[i, 1], features[i, 2]]
    for j, (m, title) in enumerate(zip(maps, ["label", "power [dB]", "residual [dB]", "EVM [dB]"])):
        axes[i, j].imshow(m.cpu(), aspect="auto", origin="lower", interpolation="nearest")
        axes[i, 0].set_ylabel(f"{name}\nPRB")
        if i == 0:
            axes[i, j].set_title(title)
for ax in axes[-1]:
    ax.set_xlabel("OFDM symbol")
plt.tight_layout()